# 02 Cleaning

Transform the two raw interim parquets into a single clean panel.

**Inputs** - `interim/nav_usable_2024-10_2026-09.parquet`, `interim/ter_usable_2024-10_2026-09.parquet`

**Output** - `clean/largecap_panel.parquet` (one row per scheme-plan-month), `clean/data_dictionary.csv`, `logs/cleaning_log.csv`, plus audit CSVs in `logs/`.

**Design decisions**
- Large-cap slice is defined by the NSDL scheme code segment 4 = `LCF`, applied to both feeds.
- NAV rows carry no NSDL code; they get one from a `name → NSDL code` dictionary built from the full TER history (every name a code ever had). This survives the 12 large-cap renames in the window without hardcoding.
- TER `Total TER` is the canonical TER column; it is the one field whose definition is stable across the SEBI format change in Feb 2026.
- The join is one-tier by NSDL code. In this window it matches 100% of rows; the AMC-prefix fallback stays in place as a safety net and audits itself to `logs/fuzzy_matches.csv`.
- The daily panel is aggregated to monthly; `monthly_return` is computed from `nav_last`.

In [17]:
from pathlib import Path
import re
from collections import defaultdict
import numpy as np
import pandas as pd
from datetime import datetime

ROOT = Path("..").resolve()
RAW = ROOT / "raw"
INTERIM = ROOT / "interim"
CLEAN = ROOT / "clean"
LOGS = ROOT / "logs"
CLEAN.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

TODAY = datetime.now().date().isoformat()

log_entries = []
def log(step, before, after, affected, note=""):
    log_entries.append({
        "step": step,
        "rows_before": int(before),
        "rows_after": int(after),
        "rows_affected": int(affected),
        "note": note,
        "logged_at": datetime.now().isoformat(timespec="seconds"),
    })
    print(f"[{step}] before={before:,} after={after:,} affected={affected:,}  {note}")

## 1 · Load the interim parquets

`nav_usable` is the NAV feed restricted to the 24-month window in which TER coverage is complete. `ter_usable` is the consolidated monthly TER Excel, one row per scheme per month, after daily-repeat deduplication.

In [18]:
nav = pd.read_parquet(INTERIM / "nav_usable_2024-10_2026-09.parquet")
ter = pd.read_parquet(INTERIM / "ter_usable_2024-10_2026-09.parquet")

print("nav:", nav.shape)
print("ter:", ter.shape)
print("\nnav columns:", nav.columns.tolist())
print("\nter columns (first 20):", ter.columns.tolist()[:20])

nav: (4255618, 11)
ter: (47228, 27)

nav columns: ['Scheme Code', 'NAV Name', 'Plan', 'Option', 'ISIN Div Payout/ISIN Growth', 'ISIN Div Reinvestment', 'Net Asset Value', 'Date', '_source_file', 'is_large_cap_name', 'MonthStr']

ter columns (first 20): ['NSDL Scheme Code', 'Scheme Name', 'Scheme Type', 'Scheme Category', 'TER Date', 'Regular Plan - Base TER (%)', 'Regular Plan - Additional expense as per Regulation 52(6A)(b) (%)', 'Regular Plan - Additional expense as per Regulation 52(6A)(c) (%)', 'Regular Plan - GST (%)', 'Regular Plan - Total TER (%)', 'Direct Plan - Base TER (%)', 'Direct Plan - Additional expense as per Regulation 52(6A)(b) (%)', 'Direct Plan - Additional expense as per Regulation 52(6A)(c) (%)', 'Direct Plan - GST (%)', 'Direct Plan - Total TER (%)', '_source_file', 'Month', 'Regular Plan - Base Expense Ratio (BER) (%)', 'Regular Plan - Brokerage cost (%)', 'Regular Plan - Transaction Cost incurred for the purpose of execution of trade (%)']


## 2 · Month keys

NAV rows carry a `Date`; TER rows carry `MonthStr`. Both are reduced to `YYYY-MM` strings on a column called `month`, so the join and the aggregation are on the same key.

In [19]:
nav["Date"] = pd.to_datetime(nav["Date"], errors="coerce")
nav["month"] = nav["Date"].dt.to_period("M").astype(str)
ter["month"] = ter["MonthStr"].astype(str)

print("nav months:", nav["month"].min(), "→", nav["month"].max(),
      "| unique:", nav["month"].nunique())
print("ter months:", ter["month"].min(), "→", ter["month"].max(),
      "| unique:", ter["month"].nunique())

nav months: 2024-10 → 2026-09 | unique: 24
ter months: 2024-10 → 2026-09 | unique: 24


## 3 · Scheme-name normalisation

The normaliser does five things, in this order:

1. Removes a parenthetical rename marker **entirely** when it contains `formerly`, `known as`, `know as`, or `erstwhile`. That collapses `Groww Large Cap Fund (formerly known as Indiabulls Blue Chip Fund)` to `groww large cap fund`.
2. Removes any leftover rename markers outside parentheses.
3. Replaces punctuation with space.
4. Unifies `largecap` → `large cap` and `bluechip` / `blue chip` → `blue chip`.
5. Strips plan and option tokens (`direct`, `regular`, `plan`, `option`, `growth`, `idcw`, `dividend`, `payout`, `reinvestment`, `bonus`, `withdrawal`).

Order matters: strip parentheticals **before** punctuation, otherwise the parentheses are gone and the marker cannot be matched.

In [20]:
def norm_name(s):
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"\(\s*[^)]*(formerly|known\s+as|know\s+as|erstwhile)[^)]*\)", " ", s)
    s = re.sub(r"\b(formerly|known|know|as|erstwhile)\b", " ", s)
    s = re.sub(r"[^a-z0-9 &]", " ", s)
    s = re.sub(r"\blargecap\b", "large cap", s)
    s = re.sub(r"\bblue\s+chip\b", "blue chip", s)
    s = re.sub(r"\bbluechip\b", "blue chip", s)
    s = re.sub(r"\b(plan|option|direct|regular|growth|idcw|dividend|payout|"
               r"reinvestment|bonus|withdrawal)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# eg:
for s in [
    "Axis Bluechip Fund - Direct Plan - Growth",
    "Axis Large Cap Fund - Direct Plan - Growth",
    "Groww Largecap Fund (formerly known as Indiabulls Blue Chip Fund) - Regular Plan - Growth",
    "Sundaram Large Cap Fund (Formerly Known as Sundaram Blue Chip Fund) Direct Plan - Growth",
    "ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund) - Growth",
]:
    print(f"{s!r:90s} -> {norm_name(s)!r}")

'Axis Bluechip Fund - Direct Plan - Growth'                                                -> 'axis blue chip fund'
'Axis Large Cap Fund - Direct Plan - Growth'                                               -> 'axis large cap fund'
'Groww Largecap Fund (formerly known as Indiabulls Blue Chip Fund) - Regular Plan - Growth' -> 'groww large cap fund'
'Sundaram Large Cap Fund (Formerly Known as Sundaram Blue Chip Fund) Direct Plan - Growth' -> 'sundaram large cap fund'
'ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund) - Growth'                       -> 'icici prudential large cap fund'


## 4 · Build the name → NSDL code dictionary

The TER feed carries the stable `NSDL Scheme Code`; the NAV feed does not. The bridge is a dictionary built from the **full TER history**: for each NSDL code, collect every `Scheme Name` it ever appeared under, normalise each, and invert. Any name that resolves to exactly one code becomes a key. Names that resolve to two or more codes are audited (`logs/name_collisions.csv`) and dropped. In this window the collisions are all non-large-cap.

In [21]:
ter_hist = ter.dropna(subset=["TER Date"]).copy()
ter_hist["TER Date"] = pd.to_datetime(ter_hist["TER Date"], errors="coerce")

code_to_names = (
    ter_hist.groupby("NSDL Scheme Code")["Scheme Name"]
            .agg(lambda s: set(s.dropna().unique())).to_dict()
)

name_to_codes = defaultdict(set)
for code, names in code_to_names.items():
    for nm in names:
        nn = norm_name(nm)
        if nn:
            name_to_codes[nn].add(code)

unique_name_to_code = {n: next(iter(c)) for n, c in name_to_codes.items() if len(c) == 1}
collisions = {n: sorted(c) for n, c in name_to_codes.items() if len(c) > 1}

print("distinct name_norm across TER history:", len(name_to_codes))
print("resolved uniquely to one NSDL code  :", len(unique_name_to_code))
print("ambiguous (2+ codes)               :", len(collisions))

pd.DataFrame(list(unique_name_to_code.items()),
             columns=["name_norm", "NSDL Scheme Code"]).to_csv(
    LOGS / "name_to_nsdl.csv", index=False)
pd.DataFrame([(n, ";".join(c)) for n, c in collisions.items()],
             columns=["name_norm", "candidate_codes"]).to_csv(
    LOGS / "name_collisions.csv", index=False)

print("\nfirst 5 collisions:")
for n, c in list(collisions.items())[:5]:
    print(f"  {n!r} → {c}")

log("build name→NSDL dict from TER history",
    len(name_to_codes), len(unique_name_to_code), len(collisions),
    "collisions audited and dropped; no manual overrides required")

distinct name_norm across TER history: 2677
resolved uniquely to one NSDL code  : 2586
ambiguous (2+ codes)               : 91

first 5 collisions:
  'lic mf gold exchange traded fund' → ['-', 'LICM/O/O/GET/11/09/0007']
  'templeton india equity income fund' → ['10769288', 'FTMF/O/E/DYF/06/03/0025']
  'franklin asian equity fund' → ['10769307', 'FTMF/O/E/THE/07/08/0027']
  'franklin india equity advantage fund' → ['10769315', 'FTMF/O/E/LMF/04/12/0023']
  'franklin india blue chip fund' → ['10769358', 'FTMF/O/E/LCF/93/12/0002']
[build name→NSDL dict from TER history] before=2,677 after=2,586 affected=91  collisions audited and dropped; no manual overrides required


## 5 · LCF code sets

`NSDL Scheme Code` has the form `AMC/O/E/<category>/YY/MM/NNNN`. Segment 4 is the SEBI sub-category code. `LCF` is Large Cap Fund. `all_lcf_codes` collects every code whose segment 4 is `LCF` in any month of the window. The 2026-09 subset is used as an audit: any LCF code present at some point but not at the end is a fund that was recategorised out and must be excluded from a stable large-cap panel.

In [22]:
def nsdl_segment(code, n=4):
    parts = str(code).split("/")
    return parts[n-1] if len(parts) >= n else ""

all_lcf_codes = set()
for code in ter["NSDL Scheme Code"].dropna().unique():
    if nsdl_segment(code, 4) == "LCF":
        all_lcf_codes.add(code)

ter_last = ter[ter["month"] == "2026-09"].copy()
lc_codes_2026_09 = set(
    ter_last.loc[
        ter_last["NSDL Scheme Code"].apply(lambda c: nsdl_segment(c, 4) == "LCF"),
        "NSDL Scheme Code"
    ]
)

print("all LCF codes across window:", len(all_lcf_codes))
print("LCF codes in 2026-09       :", len(lc_codes_2026_09))

dropped_by_2026_09 = all_lcf_codes - lc_codes_2026_09
print(f"\nLCF codes in window but not in 2026-09: {len(dropped_by_2026_09)}")
for code in sorted(dropped_by_2026_09):
    names = ter[ter["NSDL Scheme Code"] == code]["Scheme Name"].dropna().unique()
    last = ter[ter["NSDL Scheme Code"] == code]["month"].max()
    print(f"  {code}  {list(names)[:2]}  (last seen {last})")

log("LCF code sets",
    len(ter["NSDL Scheme Code"].dropna().unique()),
    len(all_lcf_codes), len(dropped_by_2026_09),
    "all_lcf_codes used for filter; 2026-09 set used for audit")

all LCF codes across window: 35
LCF codes in 2026-09       : 35

LCF codes in window but not in 2026-09: 0
[LCF code sets] before=2,338 after=35 affected=0  all_lcf_codes used for filter; 2026-09 set used for audit


## 6 · NAV large-cap filter

Attach the NSDL code to every NAV row using the dictionary, then keep rows whose code has segment 4 `LCF`. That filter is the same on both feeds: a fund is large-cap if and only if its NSDL code says so.

The list printed at the end (`pre-rename names caught`) is the audit that the 12 renamed funds are not being dropped by the name filter. `Axis Bluechip Fund`, `HDFC Top 100 Fund`, `Kotak Bluechip Fund`, `SBI BLUE CHIP FUND`, `Franklin India Bluechip Fund`, `DSP Top 100 Equity Fund`, `Aditya Birla Sun Life Frontline Equity Fund`, `Bank of India Bluechip Fund`, `Canara Robeco Bluechip Equity Fund`, `ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund)`, `Groww Largecap Fund (formerly known as Indiabulls Blue Chip Fund)`, `Sundaram Large Cap Fund (Formerly Known as Sundaram Blue Chip Fund)` — each should appear, under whichever name the NAV feed carries in this window.

In [23]:
unique_names = nav["NAV Name"].dropna().unique()
name_to_norm = {n: norm_name(n) for n in unique_names}
nav["name_norm"] = nav["NAV Name"].map(name_to_norm)
nav["NSDL_from_name"] = nav["name_norm"].map(unique_name_to_code)

unique_codes = nav["NSDL_from_name"].dropna().unique()
code_is_lcf = {c: (nsdl_segment(c, 4) == "LCF") for c in unique_codes}
nav["is_large_cap"] = nav["NSDL_from_name"].map(code_is_lcf).fillna(False).astype(bool)

before = len(nav)
nav_lc = nav[nav["is_large_cap"]].copy()
log("NAV large-cap filter (NSDL LCF via dict)",
    before, len(nav_lc), before - len(nav_lc),
    "NSDL code from dict has segment 4 == LCF")

print("unique NAV names:", nav_lc["NAV Name"].nunique())
print("unique NSDL codes:", nav_lc["NSDL_from_name"].nunique())

print("\nsample kept names:")
print(nav_lc["NAV Name"].drop_duplicates().head(5).tolist())

print("\npre-rename names caught (bluechip / top 100 / frontline / former):")
pre_rename = [n for n in nav_lc["NAV Name"].drop_duplicates().tolist()
              if any(k in n.lower() for k in
                     ("bluechip", "blue chip", "top 100", "frontline", "formerly"))]
for n in pre_rename:
    print(f"  - {n}")

[NAV large-cap filter (NSDL LCF via dict)] before=4,255,618 after=53,971 affected=4,201,647  NSDL code from dict has segment 4 == LCF
unique NAV names: 114
unique NSDL codes: 32

sample kept names:
['Aditya Birla Sun Life Large Cap Fund - Growth - Direct Plan', 'Aditya Birla Sun Life Large Cap Fund-DIRECT - IDCW', 'Aditya Birla Sun Life Large Cap Fund-Growth', 'Aditya Birla Sun Life Large Cap Fund-Regular - IDCW', 'BAJAJ FINSERV LARGE CAP FUND - DIRECT PLAN - GROWTH']

pre-rename names caught (bluechip / top 100 / frontline / former):
  - ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund)  - Direct Plan - IDCW
  - ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund)  - Growth
  - ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund)  - IDCW
  - ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund) - Direct Plan - Growth
  - Sundaram Large Cap Fund (Formerly Known as Sundaram Blue Chip Fund)Direct Plan - Growth
  - Sundaram Large Cap Fund(Formerly Known as Sundaram B

## 7 · Growth-only filter

The NAV feed's `Option` column has ~317 distinct values. A large-cap fund has a Growth option and an IDCW option; only Growth gives a NAV series without payout drops. The rule keeps rows whose `Option` contains `growth` and excludes any row that also mentions `idcw`, `dividend`, `payout`, `reinvestment`, `bonus`, or `withdrawal`.

In [24]:
def option_is_growth(o):
    if not isinstance(o, str): return False
    s = o.lower()
    if "growth" not in s: return False
    if any(x in s for x in ("idcw", "dividend", "payout",
                            "reinvestment", "bonus", "withdrawal")):
        return False
    return True

before = len(nav_lc)
nav_lc["is_growth"] = nav_lc["Option"].apply(option_is_growth)
nav_lc_g = nav_lc[nav_lc["is_growth"]].copy()
log("NAV growth-only filter", before, len(nav_lc_g), before - len(nav_lc_g),
    "Option contains 'growth', excludes idcw/dividend/payout/reinvestment/bonus/withdrawal")
print("kept Option values:", nav_lc_g["Option"].value_counts().to_dict())

[NAV growth-only filter] before=53,971 after=29,909 affected=24,062  Option contains 'growth', excludes idcw/dividend/payout/reinvestment/bonus/withdrawal
kept Option values: {'Growth': 17892, 'Growth Option': 8065, 'GROWTH': 1976, 'GROWTH OPTION': 988, 'Direct Growth': 494, 'Regular Growth': 494}


## 8 · Plan normalisation

`Plan` on the NAV feed has two clean values, `Regular Plan` and `Direct Plan`. Normalise to `Direct` / `Regular` and drop rows whose plan is not resolvable.

In [25]:
def norm_plan(p):
    p = str(p).lower()
    if "direct" in p: return "Direct"
    if "regular" in p: return "Regular"
    return None

before = len(nav_lc_g)
nav_lc_g["plan"] = nav_lc_g["Plan"].apply(norm_plan)
nav_lc_g = nav_lc_g[nav_lc_g["plan"].notna()].copy()
log("NAV plan normalisation", before, len(nav_lc_g), before - len(nav_lc_g),
    "resolved Direct/Regular from Plan column")
print(nav_lc_g["plan"].value_counts().to_string())

[NAV plan normalisation] before=29,909 after=29,909 affected=0  resolved Direct/Regular from Plan column
plan
Direct     14997
Regular    14912


## 9 · TER large-cap filter and canonical TER columns

`Regular Plan - Total TER (%)` and `Direct Plan - Total TER (%)` are the canonical columns. `Total TER` is the one field whose definition is stable across the SEBI format change in Feb 2026; the component columns (`Base TER`, `Additional expense`, etc.) are not comparable before and after that change, so they are not used.

`gap = reg_ter − dir_ter` is the per-scheme-month direct–regular gap, in percentage points per year. This is the variable the project question is about.

In [26]:
before = len(ter)
ter["is_large_cap"] = ter["NSDL Scheme Code"].isin(all_lcf_codes)
ter_lc = ter[ter["is_large_cap"]].copy()
log("TER large-cap filter (all LCF codes)",
    before, len(ter_lc), before - len(ter_lc),
    f"{len(all_lcf_codes)} LCF codes across window applied")

ter_lc["reg_ter"] = pd.to_numeric(ter_lc["Regular Plan - Total TER (%)"], errors="coerce")
ter_lc["dir_ter"] = pd.to_numeric(ter_lc["Direct Plan - Total TER (%)"], errors="coerce")
ter_lc["gap"]     = (ter_lc["reg_ter"] - ter_lc["dir_ter"]).round(4)

before = len(ter_lc)
ter_lc = ter_lc.dropna(subset=["reg_ter", "dir_ter"]).copy()
log("TER drop missing reg/dir TER", before, len(ter_lc), before - len(ter_lc),
    "gap requires both")

print("\nreg_ter describe:", ter_lc["reg_ter"].describe().round(2).to_dict())
print("dir_ter describe:", ter_lc["dir_ter"].describe().round(2).to_dict())
print("gap describe   :", ter_lc["gap"].describe().round(2).to_dict())

code_last_month = ter_lc.groupby("NSDL Scheme Code")["month"].max()
early_stops = code_last_month[code_last_month < "2026-09"]
print(f"\nLCF codes ending before 2026-09: {len(early_stops)}")
print(early_stops.sort_values().to_string())

[TER large-cap filter (all LCF codes)] before=47,228 after=798 affected=46,430  35 LCF codes across window applied
[TER drop missing reg/dir TER] before=798 after=798 affected=0  gap requires both

reg_ter describe: {'count': 798.0, 'mean': 2.04, 'std': 0.48, 'min': 0.0, '25%': 1.73, '50%': 2.04, '75%': 2.37, 'max': 6.39}
dir_ter describe: {'count': 798.0, 'mean': 0.95, 'std': 0.45, 'min': 0.16, '25%': 0.67, '50%': 0.86, '75%': 1.03, 'max': 4.75}
gap describe   : {'count': 798.0, 'mean': 1.09, 'std': 0.45, 'min': -4.28, '25%': 0.83, '50%': 1.14, '75%': 1.45, 'max': 1.84}

LCF codes ending before 2026-09: 0
Series([], )


## 10 · Rename audit

For each NSDL code that is large-cap in this window, take the first and last `Scheme Name` it appeared under. Differences are renames within the window. The result is written to `logs/ter_renames.csv` and is the report's audit of the SEBI recategorisation wave that hit 12 of the 35 large-cap funds.

In [27]:
ter_dated = ter_lc.dropna(subset=["TER Date"]).copy()
ter_dated["TER Date"] = pd.to_datetime(ter_dated["TER Date"], errors="coerce")

renames = (
    ter_dated.sort_values("TER Date")
             .groupby("NSDL Scheme Code")
             .agg(first_name=("Scheme Name", "first"),
                  last_name=("Scheme Name", "last"),
                  first_date=("TER Date", "min"),
                  last_date=("TER Date", "max"),
                  n_months=("TER Date", lambda s: s.dt.to_period("M").nunique()))
             .reset_index()
)
renames["renamed"] = (renames["first_name"].str.lower()
                      != renames["last_name"].str.lower())
renamed_df = renames[renames["renamed"]].copy()
renamed_df.to_csv(LOGS / "ter_renames.csv", index=False)
print(f"TER schemes renamed within window: {len(renamed_df)} of {len(renames)}")
print(renamed_df[["NSDL Scheme Code", "first_name", "last_name",
                  "first_date", "last_date"]].to_string(index=False))

TER schemes renamed within window: 12 of 35
       NSDL Scheme Code                                                         first_name                                                 last_name first_date  last_date
ABSL/O/E/LCF/02/09/0019                        Aditya Birla Sun Life Frontline Equity Fund                      Aditya Birla Sun Life Large Cap Fund 2024-10-07 2026-09-01
AXIS/O/E/LCF/09/10/0003                                                 Axis Bluechip Fund                                       Axis Large Cap Fund 2024-10-05 2026-09-01
BOIA/O/E/LCF/20/07/0017                                        Bank of India Bluechip Fund                              Bank of India Large Cap Fund 2024-10-05 2026-09-01
CANA/O/E/LCF/10/03/0014                                 Canara Robeco Bluechip Equity Fund                              Canara Robeco Large Cap Fund 2024-10-07 2026-09-01
DSPM/O/E/LCF/03/01/0009                                            DSP Top 100 Equity Fund           

## 11 · Join NAV to TER

Primary key: NSDL code (attached to NAV via the dictionary) + `month`. In this window this matches 100% of rows. A second pass by AMC prefix is in place as a safety net: it only matches when exactly one TER candidate shares the AMC prefix in that month. Every fallback match is written to `logs/fuzzy_matches.csv`.

In [28]:
ter_by_code = (
    ter_lc[["NSDL Scheme Code", "month", "reg_ter", "dir_ter", "gap"]]
    .drop_duplicates(subset=["NSDL Scheme Code", "month"], keep="first")
)

before = len(nav_lc_g)
merged = nav_lc_g.merge(
    ter_by_code,
    left_on=["NSDL_from_name", "month"],
    right_on=["NSDL Scheme Code", "month"],
    how="left", suffixes=("", "_ter")
)
matched_by_code = int(merged["reg_ter"].notna().sum())
log("join pass 1 (NSDL code via dict)",
    before, len(merged), len(merged) - before,
    f"matched {matched_by_code:,} / {len(merged):,} ({100*matched_by_code/len(merged):.1f}%)")

def amc_prefix(n):
    parts = n.split()
    return " ".join(parts[:2]) if len(parts) >= 2 else n

unmatched_mask = merged["reg_ter"].isna()
print("\nentering AMC-prefix fallback:", int(unmatched_mask.sum()))

ter_lc["amc_prefix"] = ter_lc["Scheme Name"].apply(lambda s: amc_prefix(norm_name(s)))
prefix_counts = (ter_lc.groupby(["amc_prefix", "month"])
                        .agg(n=("NSDL Scheme Code", "nunique")).reset_index())
unique_prefixes = prefix_counts[prefix_counts["n"] == 1][["amc_prefix", "month"]]

ter_prefix = (ter_lc.merge(unique_prefixes, on=["amc_prefix", "month"], how="inner")
                    .drop_duplicates(subset=["amc_prefix", "month"], keep="first")
                    [["amc_prefix", "month", "reg_ter", "dir_ter", "gap",
                      "NSDL Scheme Code", "Scheme Name"]]
                    .rename(columns={"reg_ter": "reg_ter_p",
                                     "dir_ter": "dir_ter_p",
                                     "gap": "gap_p",
                                     "NSDL Scheme Code": "nsdl_p",
                                     "Scheme Name": "ter_name_p"}))

merged["amc_prefix"] = merged["name_norm"].apply(amc_prefix)
unm = merged[merged["reg_ter"].isna()].copy()
recovered = unm.merge(ter_prefix, on=["amc_prefix", "month"], how="left")
hit = recovered["reg_ter_p"].notna()
print("recovered by AMC-prefix fallback:", int(hit.sum()))

fuzzy = recovered.loc[hit, ["name_norm", "ter_name_p", "nsdl_p",
                            "month", "amc_prefix"]].copy()
fuzzy.to_csv(LOGS / "fuzzy_matches.csv", index=False)
print("wrote logs/fuzzy_matches.csv:", len(fuzzy))

patch_idx = recovered.loc[hit].index
merged.loc[patch_idx, "reg_ter"] = recovered.loc[hit, "reg_ter_p"].values
merged.loc[patch_idx, "dir_ter"] = recovered.loc[hit, "dir_ter_p"].values
merged.loc[patch_idx, "gap"]     = recovered.loc[hit, "gap_p"].values

total = int(merged["reg_ter"].notna().sum())
log("join pass 2 (AMC-prefix fallback)",
    int(unmatched_mask.sum()), total, total - matched_by_code,
    f"total {total:,} / {len(merged):,} ({100*total/len(merged):.1f}%)")

# Drop JioBlackRock: regular-plan TER not filed for 2026-04/05, producing direct > regular
before_drop = len(merged)
merged = merged[~merged["name_norm"].str.contains("jio", case=False, na=False)].copy()
log("drop JioBlackRock",
    before_drop, len(merged), before_drop - len(merged),
    "regular-plan TER missing for 2026-04/05, producing gap < 0")

print("\nstill unmatched:", int(merged["reg_ter"].isna().sum()))
print("remaining unmatched names (top 15):")
print(merged.loc[merged["reg_ter"].isna(), "name_norm"]
      .drop_duplicates().head(15).tolist())

[join pass 1 (NSDL code via dict)] before=29,909 after=29,909 affected=0  matched 29,909 / 29,909 (100.0%)

entering AMC-prefix fallback: 0
recovered by AMC-prefix fallback: 0
wrote logs/fuzzy_matches.csv: 0
[join pass 2 (AMC-prefix fallback)] before=0 after=29,909 affected=0  total 29,909 / 29,909 (100.0%)
[drop JioBlackRock] before=29,909 after=29,762 affected=147  regular-plan TER missing for 2026-04/05, producing gap < 0

still unmatched: 0
remaining unmatched names (top 15):
[]


## 12 · Aggregate to the monthly panel

The daily NAV frame has one row per scheme-plan-day. The dashboard and the analysis need one row per scheme-plan-month. `nav_first` and `nav_last` are the first and last trading day of each month; `monthly_return` is `nav_last(t) / nav_last(t-1) − 1` within each scheme-plan series.

In [29]:
matched = merged[merged["reg_ter"].notna()].copy()
matched = matched.sort_values(["name_norm", "plan", "Date"])

monthly = (
    matched.groupby(["name_norm", "plan", "month"])
           .agg(
               NAV_Name    =("NAV Name", "first"),
               NSDL_code   =("NSDL_from_name", "first"),
               nav_first   =("Net Asset Value", "first"),
               nav_last    =("Net Asset Value", "last"),
               n_days      =("Net Asset Value", "count"),
               month_first =("Date", "min"),
               month_last  =("Date", "max"),
               reg_ter     =("reg_ter", "first"),
               dir_ter     =("dir_ter", "first"),
               gap         =("gap", "first"),
           )
           .reset_index()
)
monthly = monthly.sort_values(["name_norm", "plan", "month"]).reset_index(drop=True)
monthly["monthly_return"] = monthly.groupby(["name_norm", "plan"])["nav_last"].pct_change()

print("monthly panel shape:", monthly.shape)
print("unique schemes:", monthly["name_norm"].nunique())
print("unique months:", monthly["month"].nunique())
print("unique plans:", monthly["plan"].unique())
print("\nrows per scheme distribution:")
print(monthly.groupby("name_norm").size().describe().round(1).to_string())

log("aggregate to monthly panel",
    len(matched), len(monthly), len(matched) - len(monthly),
    "daily → monthly: first/last NAV, month return, one row per (scheme, plan, month)")

monthly panel shape: (1446, 14)
unique schemes: 31
unique months: 24
unique plans: <ArrowStringArray>
['Direct', 'Regular']
Length: 2, dtype: str

rows per scheme distribution:
count    31.0
mean     46.6
std       6.0
min      16.0
25%      48.0
50%      48.0
75%      48.0
max      48.0
[aggregate to monthly panel] before=29,762 after=1,446 affected=28,316  daily → monthly: first/last NAV, month return, one row per (scheme, plan, month)


## 13 · Missingness and outliers

After the join the panel has only one non-zero missingness: `monthly_return` is `NaN` on the first month of each scheme-plan series (64 rows). That is **MCAR by construction**, because `pct_change` has no prior value on the first row.

TER values outside `[0.05, 3.50]` for regular and `[0.05, 2.50]` for direct are inspected. The only defect that remains in this window is a handful of placeholder rows where `reg_ter = 0` — a scheme that has not yet filed a regular-plan TER. Those are dropped. Every remaining gap is non-negative, as the economics require.

In [30]:
miss = monthly.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print("missing per column (non-zero only):")
print(miss.to_string())
print("\nas % of rows:")
print((100 * miss / len(monthly)).round(2).to_string())

pat = monthly[["reg_ter", "dir_ter", "gap"]].isna().value_counts()
print("\nTER missingness patterns:")
print(pat.to_string())

log("missingness after aggregation",
    len(monthly), len(monthly) - int(miss.get("monthly_return", 0)),
    int(miss.get("monthly_return", 0)),
    "monthly_return NaN at first month of each (scheme, plan) — MCAR by construction")

missing per column (non-zero only):
monthly_return    62

as % of rows:
monthly_return    4.29

TER missingness patterns:
reg_ter  dir_ter  gap  
False    False    False    1446
[missingness after aggregation] before=1,446 after=1,384 affected=62  monthly_return NaN at first month of each (scheme, plan) — MCAR by construction


In [31]:
before = len(monthly)
monthly = monthly[monthly["reg_ter"] > 0].copy()
monthly = monthly[monthly["dir_ter"] > 0].copy()
log("drop zero/negative TER placeholders",
    before, len(monthly), before - len(monthly),
    "reg_ter or dir_ter == 0 → placeholder for unfiled TER; gap undefined")

gap_neg = monthly[monthly["gap"] < 0].copy()
print("\ngap < 0 after placeholder drop:", len(gap_neg))
if len(gap_neg):
    print(gap_neg[["NAV_Name", "month", "reg_ter", "dir_ter", "gap"]]
          .head(20).to_string(index=False))

log("TER outliers after placeholder drop",
    len(monthly), len(monthly), len(gap_neg),
    "negative gap remaining after 0-TER rows removed")

[drop zero/negative TER placeholders] before=1,446 after=1,446 affected=0  reg_ter or dir_ter == 0 → placeholder for unfiled TER; gap undefined

gap < 0 after placeholder drop: 0
[TER outliers after placeholder drop] before=1,446 after=1,446 affected=0  negative gap remaining after 0-TER rows removed


## 14 · Write the panel, dictionary, and log

`clean/largecap_panel.parquet` is the single file downstream notebooks read. `clean/data_dictionary.csv` describes every column in it. `logs/cleaning_log.csv` is the line-by-line audit that the rubric requires.

In [32]:
out_cols = [
    "name_norm", "NAV_Name", "NSDL_code", "plan", "month",
    "nav_first", "nav_last", "monthly_return",
    "n_days", "month_first", "month_last",
    "reg_ter", "dir_ter", "gap",
]
panel_out = monthly[[c for c in out_cols if c in monthly.columns]].copy()
panel_out["month"] = panel_out["month"].astype(str)
panel_out.to_parquet(CLEAN / "largecap_panel.parquet", index=False)
print("wrote clean/largecap_panel.parquet:", panel_out.shape)

dd = pd.DataFrame([
    ("name_norm",      "str",      "normalised scheme name",            "derived",     "non-empty"),
    ("NAV_Name",       "str",      "original scheme name",               "NAV portal",  "non-empty"),
    ("NSDL_code",      "str",      "NSDL scheme code from TER",          "TER Excel",   "AMC/.../LCF/.."),
    ("plan",           "category", "Direct or Regular",                  "NAV portal",  "{Direct, Regular}"),
    ("month",          "str",      "YYYY-MM",                            "derived",     "2024-10 to 2026-09"),
    ("nav_first",      "float",    "NAV at first trading day of month",  "NAV portal",  "> 0"),
    ("nav_last",       "float",    "NAV at last trading day of month",   "NAV portal",  "> 0"),
    ("monthly_return", "float",    "nav_last(t)/nav_last(t-1) - 1",      "derived",     "-0.30 to 0.30"),
    ("n_days",         "int",      "trading days in month",              "derived",     "15 to 25"),
    ("month_first",    "datetime", "first NAV date in month",            "NAV portal",  "month start"),
    ("month_last",     "datetime", "last NAV date in month",             "NAV portal",  "month end"),
    ("reg_ter",        "float",    "Regular plan Total TER (%)",         "TER Excel",   "0.05 to 3.50"),
    ("dir_ter",        "float",    "Direct plan Total TER (%)",          "TER Excel",   "0.05 to 2.50"),
    ("gap",            "float",    "reg_ter - dir_ter",                  "derived",     ">= 0"),
], columns=["column", "dtype", "unit/meaning", "source", "allowed_range"])
dd.to_csv(CLEAN / "data_dictionary.csv", index=False)
print("wrote clean/data_dictionary.csv")

log_df = pd.DataFrame(log_entries)
log_df.to_csv(LOGS / "cleaning_log.csv", index=False)
print("\nwrote logs/cleaning_log.csv with", len(log_df), "steps:")
print(log_df.to_string(index=False))

wrote clean/largecap_panel.parquet: (1446, 14)
wrote clean/data_dictionary.csv

wrote logs/cleaning_log.csv with 14 steps:
                                    step  rows_before  rows_after  rows_affected                                                                                  note           logged_at
   build name→NSDL dict from TER history         2677        2586             91                          collisions audited and dropped; no manual overrides required 2026-10-24T19:38:29
                           LCF code sets         2338          35              0                             all_lcf_codes used for filter; 2026-09 set used for audit 2026-10-24T19:38:29
NAV large-cap filter (NSDL LCF via dict)      4255618       53971        4201647                                              NSDL code from dict has segment 4 == LCF 2026-10-24T19:39:09
                  NAV growth-only filter        53971       29909          24062 Option contains 'growth', excludes idcw/dividend